## Introduction

This example demonstrates federated learning using the Flower framework with scikit-learn's LogisticRegression model. It shows how traditional machine learning algorithms can be adapted to work in privacy-preserving federated settings, where data remains on local devices while still contributing to a collectively trained model. Reference tutorial: https://flower.ai/docs/examples/quickstart-sklearn-tabular.html

In this implementation, we use the classic Iris dataset to train a model that classifies iris flowers into three species based on their measurements. The data is partitioned across multiple simulated clients to demonstrate how federated learning works.

When you run this example:
1. The dataset is divided among multiple clients
2. A central server initializes and distributes a global model
3. Each client trains the model on its local data partition
4. Clients share only model parameters (not raw data) with the server
5. The server aggregates these parameters using Federated Averaging
6. The process repeats for multiple rounds, improving the model with each iteration

Flower's framework handles all communication between components automatically. 

### Start by creating the Flower App

This version is interactive and should be run in a terminal.

You will be prompted to input project name, flower username, and ML framework.

In Jupyter, use the automated (non-interactive) version instead

In [1]:
# Author: Francesco Esposito (fe.digi@cbs.dk)
# scr: Flower Labs

In [2]:
# [general knowledge] INTERACTIVE VERSION (run in terminal, not in Jupyter)

# Clean up previous attempts
# rm -rf quickstart-sklearn

# Install Flower (simulation dependencies)
# python -m pip install "flwr[simulation]"

# Create the app (interactive prompts will appear)
# flwr new @flwrlabs/quickstart-sklearn

# Enter the project directory
# cd quickstart-sklearn

In [3]:
# [RUN] AUTOMATED VERSION
import os
import shutil
import subprocess
import sys

project_name = "quickstart-sklearn"
flower_username = "francesco"  # change if you want

if os.path.exists(project_name):
    shutil.rmtree(project_name)

cmd = (
    f"printf '{project_name}\n{flower_username}\n' "
    f"| flwr new @flwrlabs/quickstart-sklearn"
)

subprocess.run(cmd, shell=True, check=True)
os.chdir(project_name)
print("Current directory:", os.getcwd())


🔗 Requesting download link for @flwrlabs/quickstart-sklearn...
🔽 Downloading ZIP into memory...
📦 Unpacking into /Users/francescoesposito/Documents/TA/quickstart-sklearn...
🎊 Flower App creation successful.

To run your Flower App, first install its dependencies:

	cd quickstart-sklearn && pip install -e .

then, run the app:

 	flwr run .

💡 Check the README in your app directory to learn how to
customize it and how to run it using the Deployment Runtime.

Current directory: /Users/francescoesposito/Documents/TA/quickstart-sklearn


💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `project_name = ""` | Defines the folder name of the Flower application that will be created from the template. |
| `flower_username = ""` | Provides the Flower username requested by `flwr new`. The template uses this value to configure the generated project metadata. |
| `if loop` | Removes any previous version of the project folder so the notebook can be re-run from a clean state without conflicts. |
| `cmd = ()` | Simulates the answers that would normally be typed interactively in the terminal. |
| `subprocess.run()` | Executes the Flower command as a shell command. `check=True` stops execution if project creation fails. |
| `os.chdir()` | Moves the working directory into the generated Flower app so that later commands such as simulation run in the correct project folder. |
| `print()` | Confirms that the notebook is now operating inside the generated project directory. |


### View project structure

In [4]:
!ls -la #list all files in the current directory, along with their metadata

total 16
drwxr-xr-x@  5 francescoesposito  staff   160 Mar 25 17:36 .
drwxr-xr-x  38 francescoesposito  staff  1216 Mar 25 17:36 ..
-rw-r--r--@  1 francescoesposito  staff  2790 Mar 25 17:36 README.md
-rw-r--r--@  1 francescoesposito  staff   648 Mar 25 17:36 pyproject.toml
drwxr-xr-x@  6 francescoesposito  staff   192 Mar 25 17:36 sklearnexample


💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `!ls -la` | Lists all files in the current directory so you can verify that the Flower template was created successfully. |
| project root inspection | Helps identify the main files of the app, such as configuration files and the package directory that contains the client/server logic. |


In [5]:
!ls sklearnexample #list all files in the sklearnexample folder,where the federated learning example code is stored

__init__.py   client_app.py server_app.py task.py


💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `!ls sklearnexample` | Displays the contents of the generated Python package used in the example. |
| `sklearnexample` folder | Contains the application logic shared across the federated workflow, including the training task and the client/server apps. |
| package inspection | Helps in identifying how the federated learning app is organized before opening the individual source files. |


### Install dependencies and project
What will be installed:

- Dependencies from `pyproject.toml`:  
  These are the required packages and configurations for the project, including libraries for federated learning, machine learning, and data handling.

- `sklearnexample` package:  
  This custom package contains the federated learning example code, including key modules like:
  - `client_app.py`
  - `server_app.py`
  - `task.py`

In [6]:
!python -m pip install -e .

Obtaining file:///Users/francescoesposito/Documents/TA/quickstart-sklearn
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quickstart-sklearn (pyproject.toml) ... done
  Created wheel for quickstart-sklearn: filename=quickstart_sklearn-1.0.1-py2.py3-none-any.whl size=1166 sha256=29516394a16e62dc048d1401e37b6c4cd81315e8e8d15b150f46fdfbb6a676d4
  Stored in directory: /private/var/folders/zj/9fz4tznn7953lty4s90jjsqc0000gn/T/pip-ephem-wheel-cache-8faorfis/wheels/6b/42/3c/c4b276e9673ec65f403821c157d4aca8c3ea41bad826bf17a8
Successfully built quickstart-sklearn
  Attempting uninstall: quickstart-sklearn
    Found existing installation: quickstart-sklearn 1.0.1
    Uninstalling quickstart-sklearn-1.0.1:
      Successfully uninstalled quickstart-sklearn-1.0.1


💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `python -m pip install -e .` | Installs the current project in editable mode so the Flower app can be run directly from the notebook environment. |
| `-e` | Keeps the installation linked to the local source files, so later edits in the project folder are immediately reflected without reinstalling the package. |
| current directory `.` | Tells `pip` to install the project defined by the files in the current Flower app folder. |


### Display the contents of a .py file within the sklearnexample directory

#### 1. `task.py`: 
It provides common utilities used by both client and server.

**Key Components**:
- Model Creation: Creates a logistic regression model with initial zero parameters.
- Data Handling: Splits dataset into partitions for each client, further divided into training and testing data.
- Parameter Management: Functions to extract and update model parameters.
- Communication: Clients send model parameters to the server, which combines them and sends back updated parameters.

In [7]:
!cat sklearnexample/task.py #use the !cat command to directly view the contents of a .py file in a Jupyter Notebook.

import numpy as np
from flwr.common import NDArrays
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner
from sklearn.linear_model import LogisticRegression

# This information is needed to create a correct scikit-learn model
UNIQUE_LABELS = [0, 1, 2]
FEATURES = ["petal_length", "petal_width", "sepal_length", "sepal_width"]


def get_model_params(model: LogisticRegression) -> NDArrays:
    """Return the parameters of a sklearn LogisticRegression model."""
    if model.fit_intercept:
        params = [
            model.coef_,
            model.intercept_,
        ]
    else:
        params = [
            model.coef_,
        ]
    return params


def set_model_params(model: LogisticRegression, params: NDArrays) -> LogisticRegression:
    """Set the parameters of a sklean LogisticRegression model."""
    model.coef_ = params[0]
    if model.fit_intercept:
        model.intercept_ = params[1]
    return model


def set_initial_params(model: Log

💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `!cat sklearnexample/task.py` | Useful to inspect the shared training and evaluation utilities used by the federated app. |
| `task.py` | Usually contains the dataset loading, preprocessing, model creation, and helper functions reused by both the client and the server workflow. |
| source inspection step | Makes the generated project less of a black box by showing which parts of the code implement the machine learning task. |


### 2. `client_app.py` 
It defines how each client trains and evaluates locally.

**Key Components**:
- The Client's Job: Loads its data, trains the model, shares model parameters (not data) with the server, and tests model performance on local test data.
- The Client Class: Initializes with model and data, trains on local data, receives and updates parameters, sends updated parameters back, and reports accuracy.
- The Client Factory: Creates client instances by configuring data partitions, setting model settings, and returning a complete client ready to participate.
- The Client App: Uses `ClientApp(client_fn=client_fn)` to set up and run the Flower framework with the configured client.

In [8]:
!cat sklearnexample/client_app.py

"""sklearnexample: A Flower / sklearn app."""

import warnings

from flwr.app import ArrayRecord, Context, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp
from sklearn.metrics import log_loss

from sklearnexample.task import (
    UNIQUE_LABELS,
    create_log_reg_and_instantiate_parameters,
    get_model_params,
    load_data,
    set_model_params,
)

# Flower ClientApp
app = ClientApp()


@app.train()
def train(msg: Message, context: Context):
    """Train the model on local data."""

    # Create LogisticRegression Model
    penalty = context.run_config["penalty"]
    # Create LogisticRegression Model
    model = create_log_reg_and_instantiate_parameters(penalty)

    # Apply received pararameters
    ndarrays = msg.content["arrays"].to_numpy_ndarrays()
    set_model_params(model, ndarrays)

    # Load the data
    partition_id = context.node_config["partition-id"]
    num_partitions = context.node_config["num-partitions"]
    X_train, y_train, _, _ = load_dat

💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `!cat sklearnexample/client_app.py` | Displays the client-side application code used by each participant in the federated simulation. |
| `client_app.py` | Defines how a client receives model parameters, trains locally on its own data partition, and returns updated parameters and metrics to the server. |
| local training logic | This file is central to federated learning because clients do not share raw data; they only share model updates. |


### 3. `server_app.py` 
It defines how the server aggregates and coordinates.

**Key Components**:
- The Server's Job: Creates the initial model, decides how to combine client updates, tracks progress through training rounds, and aggregates performance metrics.
- Weighted Average Function: Combines client metrics, giving more weight to clients with larger datasets, to return a fair average.
- Server Function: Initializes the model, chooses the FedAvg strategy (Federated Averaging), sets training rounds, and configures the client participation requirements.
- The Server App: Uses `ServerApp(server_fn=server_fn)` to set up and run the Flower framework as the server.

In [9]:
!cat sklearnexample/server_app.py

"""sklearnexample: A Flower / sklearn app."""

import joblib
from flwr.app import ArrayRecord, Context
from flwr.serverapp import Grid, ServerApp
from flwr.serverapp.strategy import FedAvg

from sklearnexample.task import (
    create_log_reg_and_instantiate_parameters,
    get_model_params,
    set_model_params,
)

# Create ServerApp
app = ServerApp()


@app.main()
def main(grid: Grid, context: Context) -> None:
    """Main entry point for the ServerApp."""

    # Read run config
    num_rounds: int = context.run_config["num-server-rounds"]

    # Create LogisticRegression Model
    penalty = context.run_config["penalty"]
    model = create_log_reg_and_instantiate_parameters(penalty)
    # Construct ArrayRecord representation
    arrays = ArrayRecord(get_model_params(model))

    # Initialize FedAvg strategy
    strategy = FedAvg(fraction_train=1.0, fraction_evaluate=1.0)

    # Start strategy, run FedAvg for `num_rounds`
    result = strategy.start(
        grid=grid,
        initial

💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `!cat sklearnexample/server_app.py` | Displays the server-side orchestration code of the Flower application. |
| `server_app.py` | Defines how the server initializes the federated run, selects the aggregation strategy, and coordinates communication across rounds. |
| server role | In federated learning, the server aggregates model updates from clients to produce the new global model. |


## Run Federated Learning using Flower's simulation engine

#### 1. Run the Simulation: 
To start the federated learning simulation, run the following command `flwr run .` in your Jupyter Notebook:

In [10]:
!flwr run .

🎊 Successfully started run 6235511798805434020


💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `flwr run .` | Starts the Flower simulation using the configuration of the current project. |
| current directory `.` | Tells Flower to run the app defined in the project folder the notebook is currently inside. |
| simulation run | Launches the federated workflow in which multiple clients train locally and the server aggregates their updates over several rounds. It uses a default configuration. |


#### 2. Override Settings for ClientApp and ServerApp
You can override some of the settings defined in `pyproject.toml` for your ClientApp and ServerApp. For example, you can specify different values for parameters like regularization type, number of rounds, and other training configurations. To do this, use the --run-config option followed by the settings you want to adjust.

In [11]:
!flwr run . --run-config penalty="'l2'" #modify the penalty setting to use L2 regularization.

🎊 Successfully started run 3087455726815558558


💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `--run-config penalty="'l2'"` | Overrides the default configuration by changing the regularization type used by the scikit-learn logistic regression model. |
| `penalty='l2'` | Adds L2 regularization, which discourages very large coefficients and can improve generalization. |
| configuration override | Shows that Flower apps can be re-run with modified model settings without editing the source code directly. |


In [12]:
!flwr run . --run-config num-server-rounds=10 #modify number of rounds

🎊 Successfully started run 4096237650873111721


💡 **EXPLANATION**

| Code Element | Explanation |
|---|---|
| `--run-config num-server-rounds=10` | Overrides the number of federated rounds executed by the server during simulation. |
| `num-server-rounds` | Controls how many times the server collects updates from clients and aggregates them into a new global model. |
| more rounds | Increasing the number of rounds usually gives the global model more opportunities to improve, at the cost of more computation and communication. |
